## Strategic Patient Risk Stratification & Readmission Predictive Modeling
### Vitality Health Network (VHN) — Clinical Decision Support & Quality Improvement

**Deliverable:** Technical Appendix & Exploratory Data Analysis Notebook (`.ipynb`)  
**Role:** Health Informatics Consulting Group  
**Target Stakeholders:** Chief Medical Information Officer (CMIO), Chief Nursing Officer, Director of Quality & Care Management  

---

#### Executive & Pedagogical Context
Under the ***Hospital Readmissions Reduction Program (HRRP)*** established by the Centers for Medicare & Medicaid Services (CMS), healthcare delivery systems face substantial reimbursement penalties when 30-day all-cause readmission rates for chronic conditions exceed national risk-adjusted benchmarks. 

At ***Vitality Health Network (VHN)***, the historical 30-day readmission rate for diabetic patients has reached **18%**, far outpacing benchmark expectations and incurring millions of dollars in annual penalty exposure. VHN executive leadership has mandated the transition from reactive retrospective reporting to ***proactive, diagnostic, and prescriptive risk stratification***.

This notebook provides the complete analytical pipeline, structured across four rigorous phases:
1. ***Phase 1: Data Ingestion & Clinical Sanitation:*** Ingestion of 100,000+ patient encounters, audit of 50 features, standardizing non-standard missing values (`?`), relational key mapping (`IDs_mapping.csv`), filtering terminal/deceased records, and addressing data quality limitations.
2. ***Phase 2: Data Enrichment via Web Scraping:*** Contextualizing cryptic primary ICD-9 diagnosis codes (`diag_1`) into human-readable clinical disease descriptions using ethical, rate-limited web scraping (`requests` + `BeautifulSoup`).
3. ***Phase 3: Exploratory Data Analysis (EDA):*** Uncovering key clinical, demographic, medication, and operational drivers of readmission; evaluating class imbalance, intersectional disparities, medication regimens (Insulin vs. Oral), and discharge pathways.
4. ***Phase 4: Feature Engineering — The Vitality Complexity Index (VCI):*** Operationalizing an algorithmic bedside risk-scoring tool adapted from the validated LACE Index (Length of stay, Acuity, Comorbidities, Emergency visits) and empirically validating its predictive discrimination.

### Phase 01: Data Ingestion & Clinical Sanitation

#### Project: Patient Risk Stratification – Vitality Health Network (VHN)

#### Phase Overview

This phase focuses on preparing the raw clinical dataset for reliable
data analysis and patient risk stratification.

The main objectives of this phase are:

1. Load the clinical dataset into a Pandas DataFrame.
2. Perform an initial data quality and structure assessment.
3. Standardize non-standard missing values.
4. Analyze missing data.
5. Apply ID mappings to obtain meaningful descriptions.
6. Exclude encounters associated with expired patients.
7. Remove exact duplicate records while preserving valid patient encounters.

#### Expected Outcome

A clean and structured encounter-level dataset that can be
used for the subsequent analytical phases.

In [ ]:
# Import the required libraries
import os
import sys
import io
import time
import re
import warnings
from pathlib import Path
import csv

# Suppress minor deprecation warnings for clean reporting
warnings.filterwarnings('ignore')

# Numerical & Data Manipulation
import numpy as np
import pandas as pd

# Web Scraping Stack
import requests
from bs4 import BeautifulSoup

# Visualization Stack
import matplotlib.pyplot as plt
import seaborn as sns


print("\nRequired libraries imported successfully.")

### 1.1 Loading the Clinical Dataset

The `diabetic_data.csv` file contains patient encounter information
collected from the hospital's electronic health record system.

The dataset is loaded into a Pandas DataFrame for further processing.

The original dataset uses the question mark (`?`) to represent
missing values in certain columns.

The `na_values` parameter converts these non-standard missing-value
markers into standard NumPy `NaN` values during data ingestion.

This ensures consistent missing-value identification throughout
the data cleaning process.

In [ ]:
# Step 1.2: Define the dataset location

data_dir = Path("data_files")
raw_data_path = data_dir / "diabetic_data.csv"

if not raw_data_path.exists():
    raise FileNotFoundError(
        f"Dataset file not found: {raw_data_path.resolve()}"
    )
  

df_raw = pd.read_csv(
    raw_data_path,
    na_values=["?"],
    low_memory=False
)

print("Clinical dataset loaded successfully!")

### 1.2 Initial Data Assessment

An initial assessment is performed to understand the structure,
features, data types, and statistical characteristics of the dataset.

The following Pandas methods are used:

1. `shape` – Identifies the number of rows and columns.
2. `columns` – Displays all available feature names.
3. `info()` – Provides information about data types and non-null values.
4. `describe()` – Generates descriptive statistics for dataset features.
5. `head()` – Displays the first five records.

Special attention is given to identifier columns because their
numerical values represent categories rather than measurable quantities.

This assessment helps identify potential data quality issues
before applying further cleaning operations.

In [ ]:
# Step 1.2: Perform an initial assessment of the raw dataset

print("Dataset Dimensions:")
print(df_raw.shape)

print("\nDataset Column Names:")
print(df_raw.columns.tolist())

print("\nDataset Information:")
df_raw.info()

print("\nDescriptive Statistical Summary:")
display(df_raw.describe(include="all").T)

print("\nFirst Five Records:")
df_raw.head()

### 1.3 Loading the ID Mapping Dataset

The `IDs_mapping.csv` file contains descriptions for the
categorical identifier columns in the clinical dataset.

These mappings are required to interpret the numerical
identifiers correctly.

The following identifier columns are considered:

1. `admission_type_id`
2. `discharge_disposition_id`
3. `admission_source_id`

The mapping file is structured into separate sections.
Therefore, each section is extracted and stored as a
separate dictionary for further processing.

This approach improves the interpretability of the clinical
dataset and supports the identification of expired patients.

In [ ]:
# Step 1.3: Load and extract ID mappings

mapping_file = data_dir / "IDs_mapping.csv"


if not mapping_file.exists():
    raise FileNotFoundError(
        f"Mapping file not found: {mapping_file.resolve()}"
    )


with mapping_file.open(
    mode="r",
    encoding="utf-8-sig",
    newline=""
) as file:

    mapping_rows = list(csv.reader(file))


required_sections = [
    "admission_type_id",
    "discharge_disposition_id",
    "admission_source_id"
]


id_mappings = {}


for section in required_sections:

    mapping_dict = {}
    section_found = False

    for index, row in enumerate(mapping_rows):

        if row and row[0].strip().lower() == section:

            section_found = True

            # Read mapping records until the next section
            for mapping_row in mapping_rows[index + 1:]:

                if not mapping_row or not mapping_row[0].strip():
                    continue

                first_value = mapping_row[0].strip().lower()

                # Stop when another mapping section begins
                if first_value in required_sections:
                    break

                # Store the ID and its description
                if len(mapping_row) >= 2:
                    mapping_id = mapping_row[0].strip()
                    description = mapping_row[1].strip()

                    if mapping_id.isdigit():
                        mapping_dict[mapping_id] = description

            break

    if not section_found:
        raise ValueError(
            f"Mapping section '{section}' was not found."
        )

    id_mappings[section] = mapping_dict


for section, mapping_dict in id_mappings.items():

    print(f"\n{section}")
    print("-" * 50)

    for mapping_id, description in mapping_dict.items():
        print(f"{mapping_id}: {description}")

print("\nID mappings loaded successfully.")

### 1.4 Applying ID Mappings

The numerical identifiers in the clinical dataset represent
categorical information rather than continuous numerical values.

The extracted ID mappings are used to create additional
description columns for better data interpretation.

The following description columns are generated:

- `admission_type_description`
- `discharge_disposition_description`
- `admission_source_description`

The original identifier columns are preserved to maintain
the integrity of the raw clinical data.

These descriptions will also help identify encounters
associated with expired patients.

In [ ]:
# Step 1.4: Apply ID mappings to the clinical dataset


df_cleaned = df_raw.copy()


mapping_columns = {
    "admission_type_id": "admission_type_description",
    "discharge_disposition_id": "discharge_disposition_description",
    "admission_source_id": "admission_source_description"
}


for id_column, description_column in mapping_columns.items():

    id_values = pd.to_numeric(
        df_cleaned[id_column],
        errors="coerce"
    ).astype("Int64").astype("string")


    df_cleaned[description_column] = id_values.map(
        id_mappings[id_column]
    )


for id_column, description_column in mapping_columns.items():

    print(f"\n{id_column} and {description_column}")
    display(df_cleaned[[id_column, description_column]].drop_duplicates().head(15))
    

print("\nID descriptions applied successfully.")

### 1.5 Missing Value Analysis

Missing values can affect the accuracy and reliability
of clinical data analysis.

Although the question mark (`?`) was converted into
standard missing values during data ingestion, a detailed
missing value assessment is required.

This section calculates:

1. Total missing values in each column.
2. Percentage of missing values in each column.
3. Columns containing missing data.

The results are arranged in descending order based on
the percentage of missing values.

This analysis helps identify columns that require
special handling before further data processing.

In [ ]:
# Step 1.5: Analyze missing values in the dataset

# Calculate the total missing values in each column
missing_counts = df_cleaned.isnull().sum()

# Calculate the percentage of missing values in each column
missing_percentages = (
    df_cleaned.isnull().mean() * 100
)


missing_summary = pd.DataFrame({
    "Missing Count": missing_counts,
    "Missing Percentage (%)": missing_percentages
})

# Sort columns by missing percentage
missing_summary = missing_summary.sort_values(
    by="Missing Percentage (%)",
    ascending=False
)


print("Missing Value Analysis:")

display(
    missing_summary[
        missing_summary["Missing Count"] > 0
    ]
)


print(
    "\nTotal Missing Values:",
    df_cleaned.isnull().sum().sum()
)

### 1.6 Weight Feature Analysis

The `weight` column is examined to determine the extent
of missing information.

A missing value percentage greater than 90% indicates
that the feature contains insufficient information for
reliable analysis.

According to the project requirements, if more than 90%
of the values are missing, the entire `weight` column
must be removed.

This decision is documented as a Data Quality Limitation
because removing the feature results in the loss of
potentially useful clinical information.

If the missing percentage does not exceed 90%, the
column is retained for further consideration.

In [ ]:
# Step 1.6: Analyze and handle missing values in the weight column


if "weight" in df_cleaned.columns:

    weight_missing_percentage = (
        df_cleaned["weight"].isnull().mean() * 100
    )

    print("Weight Column Analysis")
    print("-" * 40)

    print(
        "Total Records:",
        len(df_cleaned)
    )

    print(
        "Missing Values:",
        df_cleaned["weight"].isnull().sum()
    )

    print(
        f"Missing Percentage: "
        f"{weight_missing_percentage:.2f}%"
    )

    # Apply the 90% missing value threshold
    if weight_missing_percentage > 90:

        df_cleaned.drop(
            columns=["weight"],
            inplace=True
        )

        print("\nData Quality Limitation:")
        print(
            "The weight column was removed because "
            "more than 90% of its values were missing."
        )

    else:

        print(
            "\nThe weight column is retained because "
            "its missing percentage does not exceed 90%."
        )

else:

    print(
        "The weight column does not exist in the dataset."
    )

### 1.7 Gender Data Sanitation & Demographic Validation

The `gender` feature represents a critical demographic attribute required for clinical risk stratification, epidemiological profiling, and health equity analysis.

An initial audit of the demographic distribution reveals non-standard entries categorized as `'Unknown/Invalid'`. These values lack clinical validity and demographic utility for downstream predictive modeling.

This step performs the following data sanitation tasks:

1. Inspect the frequency distribution of gender categories before cleaning.
2. Standardize non-valid demographic entries (`'Unknown/Invalid'`) into standard `NaN` values to ensure consistent missing-value tracking.
3. Validate the post-cleaning category distribution and verify the total count of missing values.

This process ensures that invalid entries do not distort demographic summaries, feature encodings, or disparity assessments in subsequent analytical phases.

In [ ]:
# Step 1.7: Standardize invalid entries in the gender column

print("Before cleaning:")
print(df_cleaned['gender'].value_counts(dropna=False))


df_cleaned['gender'] = df_cleaned['gender'].replace('Unknown/Invalid', np.nan)


print("\nAfter cleaning:")
print(df_cleaned['gender'].value_counts(dropna=False))


print("\nMissing gender values:", df_cleaned['gender'].isnull().sum())

### 1.8 Handling Deceased Patients

Patient encounters associated with expired patients must be
excluded before performing readmission analysis.

The `discharge_disposition_id` column contains identifiers
representing the patient's discharge destination or status.

The corresponding descriptions from `IDs_mapping.csv` are
used to identify discharge records associated with expired patients.

All discharge disposition descriptions containing the word
"Expired" are identified.

The corresponding patient encounters are removed from the
working dataset.

The original raw dataset is preserved, and the cleaning
operation is performed on `df_cleaned`.

This step helps prevent deceased patients from being included
in subsequent readmission analysis.

In [ ]:
# Step 1.7: Identify and filter deceased/terminal patients

# Target codes: 11, 19, 20, 21
expired_codes = [11, 19, 20, 21]

deceased_mask = df_cleaned['discharge_disposition_id'].isin(expired_codes)
num_deceased = deceased_mask.sum()
initial_count = len(df_cleaned)

df_cleaned = df_cleaned[~deceased_mask].copy()

records_after_removal = len(df_cleaned) 

print("\nExpired Patient Removal Summary")
print("-" * 45)

print("Records Before Removal:", initial_count)
print("Expired Encounters Removed:", num_deceased)
print("Records After Removal:", records_after_removal)


print("\nExpired patient encounters handled successfully.")

### 1.9 Removing Duplicate Records

Duplicate records can affect the accuracy and reliability
of clinical data analysis.

The dataset is examined to identify exact duplicate rows
using all available columns.

Only records that are completely identical are removed.

Multiple encounters belonging to the same patient are valid
clinical records and must be preserved.

Therefore, deduplication is performed using the entire
record rather than the patient identifier (`patient_nbr`).

The number of duplicate records is calculated before removal,
and the dataset is checked again after the cleaning process.

In [ ]:
# Step 1.9: Identify and remove exact duplicate records

duplicate_count = df_cleaned.duplicated().sum()

print("Duplicate Records Analysis")
print("-" * 40)

print("Records Before Deduplication:", len(df_cleaned))
print("Exact Duplicate Records Found:", duplicate_count)


df_cleaned = df_cleaned.drop_duplicates().copy()

print("\nDeduplication Summary")
print("-" * 40)

print("Records After Deduplication:", len(df_cleaned))

if duplicate_count > 0:
    df_cleaned.drop_duplicates(inplace=True)
    print("Exact duplicates dropped.")

unique_patients = df_cleaned['patient_nbr'].nunique()
print(f"Unique Individual Patients: {unique_patients:,}")

# Distribution of repeat encounters
repeat_counts = df_cleaned['patient_nbr'].value_counts()
print(f"Patients with multiple hospitalizations: {(repeat_counts > 1).sum():,} ({(repeat_counts > 1).sum() / unique_patients * 100:.2f}%)")

print("\nExact duplicate records handled successfully.")

### 1.10 Schema Standardization & Categorical Type Conversion

During initial ingestion, columns containing numeric identifiers (such as `admission_type_id`, `discharge_disposition_id`, and `admission_source_id`) are parsed as integers (`int64`). 

Treating categorical codes as numeric measurements is clinically and statistically invalid:
1. **Mathematical Inappropriateness:** Arithmetic operations (such as computing mean or standard deviation on admission or discharge codes) are meaningless.
2. **Algorithm Distortion:** Machine learning algorithms would misinterpret these discrete categories as continuous numerical scales with intrinsic ordinal rank.
3. **Demographic & Operational Attributes:** Key clinical features (`race`, `gender`, `age`, `payer_code`, `medical_specialty`) represent discrete nominal/ordinal factors that should be formally defined as categories.

In this step, we:
* Enable full column visibility across the Pandas session.
* Convert numerical identifier columns and demographic features into the Pandas `category` data type.
* Verify the updated schema to ensure proper representation and optimized memory utilization.

In [ ]:
# Step 1.10: Convert identifier and demographic columns to categorical data types

print("Show all columns before converted")
print("-" * 50)
print(f"Total columns: {len(df_cleaned.columns)}\n")

with pd.option_context('display.max_rows', None):
    display(df_cleaned.dtypes.to_frame(name='Data Type'))


categorical_columns = [
    'admission_type_id',
    'discharge_disposition_id',
    'admission_source_id',
    'race',
    'gender',
    'age',
    'payer_code',
    'medical_specialty'
]


for column in categorical_columns:
    df_cleaned[column] = df_cleaned[column].astype('category')


print("Categorical Feature Conversion Summary")
print("-" * 50)
print(f"Total categorical columns converted: {len(categorical_columns)}\n")


display(df_cleaned[categorical_columns].dtypes.to_frame(name="Updated Data Type"))

### 1.11 Identifying the Oral Medication Cohort

According to Section 3.2 and Phase 1 specifications of the project blueprint, the clinical database contains 22 supplementary medication features recorded as individual diabetic drugs (distinct from `insulin`).

These 22 oral agents encompass sulfonylureas, biguanides, thiazolidinediones, meglitinides, alpha-glucosidase inhibitors, and combination therapies:
* **Single Agents:** `metformin`, `repaglinide`, `nateglinide`, `chlorpropamide`, `glimepiride`, `acetohexamide`, `glipizide`, `glyburide`, `tolbutamide`, `pioglitazone`, `rosiglitazone`, `acarbose`, `miglitol`, `troglitazone`, `tolazamide`, `examide`, `citoglipton`.
* **Combination Formulations:** `glyburide-metformin`, `glipizide-metformin`, `glimepiride-pioglitazone`, `metformin-rosiglitazone`, `metformin-pioglitazone`.

In this step, we:
1. Define and verify the presence of all 22 oral medication attributes in `df_cleaned`.
2. Confirm each feature records medication dosage adjustments (`"No"`, `"Steady"`, `"Up"`, `"Down"`).
3. Isolate the list of oral medication columns to support comparative efficacy analysis against Insulin therapy in Phase 3.

In [ ]:
# Step 1.11: Identify, verify, and isolate the 22 oral medication features

# Define the complete list of 22 oral medication columns
oral_medication_columns = [
    "metformin", "repaglinide", "nateglinide", "chlorpropamide", "glimepiride",
    "acetohexamide", "glipizide", "glyburide", "tolbutamide", "pioglitazone",
    "rosiglitazone", "acarbose", "miglitol", "troglitazone", "tolazamide",
    "examide", "citoglipton", "glyburide-metformin", "glipizide-metformin",
    "glimepiride-pioglitazone", "metformin-rosiglitazone", "metformin-pioglitazone"
]

# Verify column availability in the cleaned dataset
available_oral_meds = [
    col for col in oral_medication_columns 
    if col in df_cleaned.columns
]

missing_oral_meds = [
    col for col in oral_medication_columns 
    if col not in df_cleaned.columns
]

print("Oral Medication Cohort Audit")
print("-" * 50)
print(f"Expected Oral Medications : {len(oral_medication_columns)}")
print(f"Identified in Dataset     : {len(available_oral_meds)}")
print(f"Missing Columns           : {len(missing_oral_meds)}")

if len(available_oral_meds) == 22:
    print("\nVerification Passed: All 22 oral medication features successfully verified.")
else:
    print(f"\nWarning: Missing medication features: {missing_oral_meds}")




print("\nDose Level Distribution for Oral Agents:")
display(df_cleaned[available_oral_meds].apply(pd.Series.value_counts).fillna(0).astype(int))

### 1.12 Final Dataset Assessment

In this step, the cleaned clinical dataset is reviewed to verify the results of the data sanitation process.

The final dataset dimensions, missing values, duplicate records, and data types are examined.

In [ ]:
# Step 1.12: Perform final verification

print("Final Dataset Shape:", df_cleaned.shape)


print("\nRemaining Missing Values:")
print(df_cleaned.isnull().sum().sum())

print("\nRemaining Duplicate Rows:", df_cleaned.duplicated().sum())


print("\nFinal Dataset Information:")
df_cleaned.info()

display(df_cleaned.head())

### Phase 1 Conclusion: Data Ingestion & Clinical Sanitation Summary

Phase 1 established the clinical and technical foundation for the Vitality Health Network (VHN) patient readmission predictive modeling pipeline. Starting with 101,766 raw inpatient encounters, the dataset underwent a structured data hygiene protocol aligned with Centers for Medicare & Medicaid Services (CMS) hospital quality standards.

#### Key Deliverables & Methodological Milestones Achieved:
1. **Missing Data Standardization:** 
   * Non-standard missing indicators (`"?"`) were converted into standard `NaN` representations.
   * The `weight` feature exhibited **96.86% missingness** (98,569 missing encounters). In accordance with the project specification threshold (>90%), `weight` was completely pruned from the feature space and formally cited as a **Data Quality Limitation** to prevent the introduction of severe synthetic imputation bias.
2. **Relational Contextualization:**
   * Parsed relational keys from `IDs_mapping.csv` to map human-readable clinical descriptions for `admission_type_id`, `discharge_disposition_id`, and `admission_source_id`.
3. **Mitigation of Survival Bias:**
   * Identified and filtered **1,652 expired patient encounters** (discharge disposition codes `11`, `19`, `20`, `21`). Deceased patients cannot experience an unplanned 30-day readmission; retaining them would introduce survival bias and artificially deflate predicted readmission rates.
4. **Data Hygiene & Demographic Sanitation:**
   * Audited and removed exact duplicate records while maintaining valid multiple encounters per unique patient (`patient_nbr`).
   * Standardized invalid gender entries (`Unknown/Invalid`) to maintain demographic integrity.
   * Converted categorical identifier codes into Pandas `category` types to prevent mathematical misinterpretation by predictive algorithms.
5. **Therapeutic Cohort Isolation:**
   * Cataloged and verified all **22 oral diabetic medications**, establishing the benchmark cohort for comparative efficacy testing against insulin therapy.